<a href="https://colab.research.google.com/github/A-ghori/sklearn/blob/main/MAE_MSE_R2_Score_Ridge_(L2_)_Lasso_(L1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Part 1: The Error Metrics (MAE vs. MSE)
Both MAE and MSE measure the gap between the true values (y) and predicted values (
y^
​
 ). Their difference lies in how they punish mistakes.

True value: y = 20
Prediction: ŷ = 10  -->  Error = |20 - 10| = 10

MAE Penalty : 10
MSE Penalty : 10² = 100  (Explodes 10x larger!)

#MAE
Scale: Expressed in the exact same units as the target (e.g., MAE=2.26 mpg means off by 2.26 mpg on average).
Behavior: Treats small errors and large errors linearly. A 10 mpg error is penalized exactly 10× more than a 1 mpg error.
Best used when: Your dataset has rare anomalies or extreme outliers that you do not want dominating the model.

#MSE (Mean Squared Error) & RMSE
Scale: MSE is in squared units (mpg
2
 ); taking the square root (RMSE) brings it back to original units (mpg).
Behavior: Heavily penalizes large mistakes because squaring amplifies large numbers (0.5
2
 =0.25, but 10
2
 =100).
Best used when: A large mistake is catastrophic in the real world (e.g., predicting hospital bed capacity or airplane fuel) and must be actively prevented.

#Part 2: The Benchmark Metric (R^2Score)

While MAE and RMSE tell you the absolute error size, they cannot tell you if a model is "good" or "bad" across different datasets. R^2 solves this by comparing your model against a naive mean baseline.


So take an example like Explaindes by Model(ss_reg) -> (84.75%) so error left(ss_res) -> (15.25%)
so R^2 = 0.8475 (Model will eliminated 84.75% of baseline variance)

# OK IMPORTANT CHECKS FOR THIS I DONT DO THE WHOLE EDA PROCESS IN THIS FILE BECAUSE I DID THiS BEFORE IN MPG FILE IF YOU HAD DOUBT YOU CAN CHECKOUT MY MPG FILE I ONLY DEMONSTRATE THE WORKDONE OF MAE MSE AND R^2 SCORE

In [37]:
import pandas as pd
import numpy as np

np.random.seed(100) #"Every time this code runs, reset the random number sequence back to position 100."

n_samples = 100
x1 = np.random.uniform(0, 10, n_samples) #np.random.uniform(0, 10)	Flat Line	0 se 10 ke beech har number aane ka chance barabar hota hai. (Jaise dice phekna).
x2 = np.random.uniform(0,10, n_samples)
#np.random.normal(loc=0, scale=1.5, size=100)  ka seedha matlab hai: aise random numbers generate karo jo 0 ke aas-paas zyada ho aur 0 se door (bahut bade ya bahut chhote) bahut kam ho.
noise = np.random.normal(0,1.30,n_samples) # normal means normal distribution

y = 4.5 * x1 - 2.0 * x2 + 10 + noise

print(y)
print(noise)

print(x1)
print(x2)


[21.10214926  5.99585836 18.31852116 41.5365459  -3.75391788 -2.42791289
 28.6562145  25.42294682 -0.85196506 30.66365025 49.81524821 17.43337365
 -2.74991196 -4.29743969 16.0581831  44.63719535 35.82734453 11.41033033
 37.35401973 10.99744307 20.09946748 37.95867305 35.73454208 23.33100348
  9.33070583 16.65322453  2.74747049 12.85096409 37.40050674  1.48357105
 34.22355254 30.9311582  -3.17980239 27.22318962 13.15024042 45.07270985
 44.01624024 -4.41944066 29.62902414 16.94214483 40.98340337 26.39109187
 24.92318401  9.50577949  1.3777417  28.49447826 25.40116757  9.05922527
 19.33203456 42.32172139 51.88570399 39.34697626 14.65914225 22.17611964
 16.91355275 18.63016959  7.08503706 10.00339154  5.35525914 25.24101118
 23.31771011 27.44369799 30.12393291  0.59808506 36.57513023 41.69009447
 30.10343485 21.00011087 12.83800489  3.12272533 19.37134242 18.29054685
  9.93042618 49.02434021 32.40980459 19.84274752 31.73167216 19.67423441
 11.55882662 10.49241676 20.21380237 -2.01363407 23

In [38]:
X = pd.DataFrame({"Feature_1":x1, "Feature_2":x2 })
print(X)
print(f"y", y)
print(y[:5].round(2))

    Feature_1  Feature_2
0    5.434049   7.782892
1    2.783694   7.795984
2    4.245176   6.103282
3    8.447761   3.090003
4    0.047189   6.977349
..        ...        ...
95   6.589401   6.321921
96   2.542575   7.329950
97   6.411013   9.024095
98   2.001236   1.622469
99   6.576248   4.058813

[100 rows x 2 columns]
y [21.10214926  5.99585836 18.31852116 41.5365459  -3.75391788 -2.42791289
 28.6562145  25.42294682 -0.85196506 30.66365025 49.81524821 17.43337365
 -2.74991196 -4.29743969 16.0581831  44.63719535 35.82734453 11.41033033
 37.35401973 10.99744307 20.09946748 37.95867305 35.73454208 23.33100348
  9.33070583 16.65322453  2.74747049 12.85096409 37.40050674  1.48357105
 34.22355254 30.9311582  -3.17980239 27.22318962 13.15024042 45.07270985
 44.01624024 -4.41944066 29.62902414 16.94214483 40.98340337 26.39109187
 24.92318401  9.50577949  1.3777417  28.49447826 25.40116757  9.05922527
 19.33203456 42.32172139 51.88570399 39.34697626 14.65914225 22.17611964
 16.91355275 18.6

In [39]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train, y_test = train_test_split(X,y, test_size=0.20, random_state=42)

from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

print(f"Learned Intercept (True = 10.0) : {lr.intercept_:.2f}")
print(f"Learned w1 for X1 (True = 4.5)  : {lr.coef_[0]:.2f}")
print(f"Learned w2 for X2 (True = -2.0) : {lr.coef_[1]:.2f}")

Learned Intercept (True = 10.0) : 10.15
Learned w1 for X1 (True = 4.5)  : 4.44
Learned w2 for X2 (True = -2.0) : -1.93


# Make Predictions on Unseen Data 20 test samples


In [40]:
y_pred = lr.predict(X_test)

print(y_pred)
import pandas as pd
comparison = pd.DataFrame(
    {
        "True Value" : y_test[:5].round(2),
        "Predicted Value" : y_pred[:5].round(2),
        "Difference (Residual)": (y_test[:5] - y_pred[:5]).round(2)
    }
)

print(comparison)

[49.33329967 22.88492129 20.23587881 29.64800185  1.25732949 16.57044938
 36.27591215 19.06794642 49.25445003 19.23237509 37.47472573 34.97569127
 49.95864197 26.58352725 32.59191392 -3.11863042 32.55572219 21.03680306
  0.53708348 31.65949297]
   True Value  Predicted Value  Difference (Residual)
0       49.18            49.33                  -0.16
1       22.18            22.88                  -0.71
2       19.37            20.24                  -0.86
3       28.49            29.65                  -1.15
4        1.38             1.26                   0.12


#Calculate Regression Accuracy (R^2, MAE, MSE, RMSE)

In [41]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("--- Final Model Performance ---")
print(f"R² Score : {r2:.4f}  ({r2 * 100:.2f}% variance explained)")
print(f"MAE      : {mae:.2f}  (Average error per prediction)")
print(f"MSE      : {mse:.2f}")
print(f"RMSE     : {rmse:.2f}")

--- Final Model Performance ---
R² Score : 0.9945  (99.45% variance explained)
MAE      : 0.89  (Average error per prediction)
MSE      : 1.27
RMSE     : 1.13
